# Amazon last-mile route optimizer

Capacitated vehicle routing for the first 100 orders in the Amazon delivery dataset.

The baseline and the optimized routes are measured on the **same cleaned stops** and the **same depot**. Both totals are geodesic miles and both include the drive back to the depot. Invalid GPS points are removed before either distance is computed.


In [1]:
%pip install -q ortools folium geopy pandas numpy


Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import folium
from geopy.distance import geodesic
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

# First 100 orders. Ten of them have null-island coordinates and are dropped below.
SAMPLE_SIZE = 100
# Customer stops per van. The depot does not count toward this cap.
MAX_STOPS_PER_VAN = 15
# Vehicles the solver may use. It leaves a van idle when that shortens the routes.
FLEET_SIZE = 15
# Stated operating-cost assumption from the project write-up.
COST_PER_MILE = 1.20
DAYS_PER_YEAR = 365
# Guided local search plateaued by ~35s on this instance; 45s is past that.
SOLVE_SECONDS = 45
# Bounding box used to reject coordinates that are not in India.
INDIA_LAT = (8.0, 37.0)
INDIA_LON = (68.0, 89.0)
DEPOT = 0


## Load the sample

`amazon_delivery.csv` is the dataset the original notebook read from `archive.zip` (that zip was never committed). The first order id in the saved notebook output is `ialx566343618`.


In [3]:
raw = pd.read_csv("amazon_delivery.csv")
assert raw.iloc[0]["Order_ID"] == "ialx566343618"

sample = raw.head(SAMPLE_SIZE).copy()
sample = sample.rename(columns={"Drop_Latitude": "lat", "Drop_Longitude": "lon"})
print(f"Loaded {len(raw):,} orders; using the first {len(sample)}")
print(
    f"Sample dates: {sample['Order_Date'].min()} to {sample['Order_Date'].max()} "
    f"({sample['Order_Date'].nunique()} distinct dates)"
)
sample[["Order_ID", "lat", "lon", "Order_Date", "Vehicle"]].head()


Loaded 43,739 orders; using the first 100
Sample dates: 2022-02-11 to 2022-04-05 (40 distinct dates)


,Order_ID,lat,lon,Order_Date,Vehicle
0,ialx566343618,22.765049,75.912471,2022-03-19,motorcycle
1,akqg208421122,13.043041,77.813237,2022-03-25,scooter
2,njpu434582536,12.924264,77.688400,2022-03-19,motorcycle
3,rjto796129700,11.053669,77.026494,2022-04-05,motorcycle
4,zguw716275638,13.012793,80.289982,2022-03-26,scooter


## Clean coordinates, then place the depot

The ten bad stops in this sample are not scattered ocean readings. Each one is a null-island GPS failure: the drop is near `(0, 0)` (Gulf of Guinea) and the store is exactly `(0, 0)`. They are removed **before** the baseline and **before** the depot is placed.

The depot is the average latitude and longitude of the stops that remain. That same point is the start and end of the baseline tour, node 0 in the solver, and the marker on both maps. It is added after filtering so a cleaning rule cannot delete it and silently turn a customer into the depot.


In [4]:
inside_india = (
    sample["lat"].between(*INDIA_LAT)
    & sample["lon"].between(*INDIA_LON)
    & sample["lat"].notna()
    & sample["lon"].notna()
)
removed = sample.loc[~inside_india, ["Order_ID", "lat", "lon", "Store_Latitude", "Store_Longitude"]].copy()
stops = sample.loc[inside_india].reset_index(drop=True)

# These ten points are the whole of the failure in this sample: every rejected
# coordinate is near (0, 0), and every kept coordinate is inside the box.
assert len(removed) == 10
assert ((removed["lat"].abs() < 1) & (removed["lon"].abs() < 1)).all()
assert stops["lat"].between(*INDIA_LAT).all()
assert stops["lon"].between(*INDIA_LON).all()

print(f"Rows before cleaning: {len(sample)}")
print(f"Rows after cleaning:  {len(stops)}")
print(f"Removed {len(removed)} null-island points:")
print(removed.to_string(index=False))

neg_store = sample.loc[sample["Store_Latitude"] < 0, "Order_ID"].tolist()
print(
    f"Store coordinates are not used for routing. "
    f"{len(neg_store)} order(s) in the sample have a negative store latitude "
    f"({', '.join(neg_store) or 'none'}); the drop is inside India and is kept."
)

depot_lat = float(stops["lat"].mean())
depot_lon = float(stops["lon"].mean())
print(f"Depot (average of the {len(stops)} cleaned stops): ({depot_lat:.6f}, {depot_lon:.6f})")


Rows before cleaning: 100
Rows after cleaning:  90
Removed 10 null-island points:
     Order_ID  lat  lon  Store_Latitude  Store_Longitude
zotl583816092 0.11 0.11             0.0              0.0
dcdm719150402 0.03 0.03             0.0              0.0
bbow373480117 0.08 0.08             0.0              0.0
etss702593505 0.06 0.06             0.0              0.0
emlt327861941 0.02 0.02             0.0              0.0
rufz727804632 0.04 0.04             0.0              0.0
lgee738117860 0.08 0.08             0.0              0.0
nxfs257449148 0.02 0.02             0.0              0.0
bhdi726355812 0.05 0.05             0.0              0.0
dfit633531471 0.11 0.11             0.0              0.0
Store coordinates are not used for routing. 1 order(s) in the sample have a negative store latitude (hnme204254454); the drop is inside India and is kept.
Depot (average of the 90 cleaned stops): (18.368161, 76.863512)


## One distance table for both totals

Pairwise distance is `geopy.distance.geodesic` on the WGS-84 ellipsoid.

- The **miles** table is what gets reported, for the baseline and for the solved routes.
- The **meters** table is the integer cost matrix OR-Tools requires. Each entry is rounded to the nearest meter (not truncated).

Node 0 is the depot. Nodes 1..N are the cleaned stops in their original file order.


In [5]:
coordinates = [(depot_lat, depot_lon)] + list(zip(stops["lat"], stops["lon"]))
n = len(coordinates)
assert n == len(stops) + 1

miles = np.zeros((n, n))
meters = np.zeros((n, n), dtype=np.int64)
for i in range(n):
    for j in range(i + 1, n):
        leg = geodesic(coordinates[i], coordinates[j])
        miles[i, j] = miles[j, i] = leg.miles
        meters[i, j] = meters[j, i] = int(round(leg.meters))

def tour_miles(nodes):
    """Geodesic miles along a node sequence, including every hop that is listed."""
    return float(sum(miles[nodes[i], nodes[i + 1]] for i in range(len(nodes) - 1)))

# Original order, starting at the depot and returning to it.
# The return hop is included because every optimized route returns to the depot.
baseline_route = list(range(n)) + [DEPOT]
baseline_miles = tour_miles(baseline_route)
print(f"Nodes in both problems: {len(stops)} stops + 1 depot")
print(f"Baseline unoptimized distance: {baseline_miles:,.1f} miles")
print("(Closed tour: depot, then the cleaned stops in original order, then back to the depot.)")


Nodes in both problems: 90 stops + 1 depot
Baseline unoptimized distance: 45,517.9 miles
(Closed tour: depot, then the cleaned stops in original order, then back to the depot.)


## Unoptimized map

Gray line: the baseline tour on the cleaned stops. It uses the same depot as the solver and closes back to that depot.


In [6]:
def make_map():
    m = folium.Map(
        location=[depot_lat, depot_lon],
        zoom_start=6,
        tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
        attr="Google",
    )
    folium.Marker(
        [depot_lat, depot_lon],
        popup="Depot",
        icon=folium.Icon(color="blue", icon="home"),
    ).add_to(m)
    return m

before_map = make_map()
for stop_index, stop in stops.iterrows():
    node = stop_index + 1  # node 0 is the depot
    folium.CircleMarker(
        [stop["lat"], stop["lon"]],
        radius=4,
        color="#c0392b",
        fill=True,
        fill_opacity=0.9,
        popup=f"{stop['Order_ID']} (stop {node})",
    ).add_to(before_map)

before_points = [[coordinates[i][0], coordinates[i][1]] for i in baseline_route]
folium.PolyLine(before_points, color="#7f8c8d", weight=3, opacity=0.85).add_to(before_map)
before_map


## Capacitated VRP

Google OR-Tools. Each customer has demand 1 and the depot has demand 0, so the capacity is a cap on **customer stops**, not on stops-plus-depot. A second dimension is not registered: adding another dimension with the same name fails and used to leave the depot counted inside the cap.

The search builds a global cheapest-arc solution and then runs guided local search. The original notebook stopped at the first `PATH_CHEAPEST_ARC` solution and never ran a local-search metaheuristic.


In [7]:
manager = pywrapcp.RoutingIndexManager(n, FLEET_SIZE, DEPOT)
routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return int(meters[from_node, to_node])

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

def demand_callback(from_index):
    return 0 if manager.IndexToNode(from_index) == DEPOT else 1

demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index,
    0,  # no slack
    [MAX_STOPS_PER_VAN] * FLEET_SIZE,
    True,  # cumul starts at 0
    "Stops",
)

search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.GLOBAL_CHEAPEST_ARC
)
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_parameters.time_limit.FromSeconds(SOLVE_SECONDS)
search_parameters.log_search = False

print(
    f"Optimizing {len(stops)} stops, fleet of {FLEET_SIZE}, "
    f"at most {MAX_STOPS_PER_VAN} stops per van, {SOLVE_SECONDS}s guided local search..."
)
solution = routing.SolveWithParameters(search_parameters)
if solution is None:
    raise RuntimeError(f"No feasible route. Solver status: {routing.status()}")

routes = []
for vehicle_id in range(FLEET_SIZE):
    if not routing.IsVehicleUsed(solution, vehicle_id):
        continue
    index = routing.Start(vehicle_id)
    route = []
    while True:
        route.append(manager.IndexToNode(index))
        if routing.IsEnd(index):
            break
        index = solution.Value(routing.NextVar(index))
    routes.append(route)

# Same stops, each exactly once, every route closed at the depot, cap honored.
served = []
for route in routes:
    assert route[0] == DEPOT and route[-1] == DEPOT
    customers = route[1:-1]
    assert DEPOT not in customers
    assert 1 <= len(customers) <= MAX_STOPS_PER_VAN
    served.extend(customers)
assert sorted(served) == list(range(1, n))

optimized_miles = sum(tour_miles(route) for route in routes)
objective_miles = solution.ObjectiveValue() / 1609.344
# Rounding each matrix entry to a meter should move the total by well under a mile.
assert abs(optimized_miles - objective_miles) < 1.0

vans_used = len(routes)
print(f"Vans used: {vans_used}")
for van_number, route in enumerate(routes, start=1):
    print(
        f"  Van {van_number}: {len(route) - 2} stops, "
        f"{tour_miles(route):,.2f} miles"
    )
print(f"Optimized distance (geodesic miles): {optimized_miles:,.1f}")
print(f"Solver objective converted from rounded meters: {objective_miles:,.1f}")


Optimizing 90 stops, fleet of 15, at most 15 stops per van, 45s guided local search...


Vans used: 6
  Van 1: 15 stops, 720.83 miles
  Van 2: 15 stops, 1,180.07 miles
  Van 3: 15 stops, 833.41 miles
  Van 4: 15 stops, 994.18 miles
  Van 5: 15 stops, 1,916.95 miles
  Van 6: 15 stops, 2,212.66 miles
Optimized distance (geodesic miles): 7,858.1
Solver objective converted from rounded meters: 7,858.1


## Optimized map

Each color is one van. Every line returns to the depot, which is the distance the total includes.


In [8]:
route_colors = [
    "#1b9e77", "#d95f02", "#7570b3", "#e7298a", "#66a61e",
    "#e6ab02", "#a6761d", "#1f78b4", "#b2df8a", "#fb9a99",
    "#6a3d9a", "#b15928", "#17becf", "#8c564b", "#bcbd22",
]
after_map = make_map()
for route, color in zip(routes, route_colors):
    points = [[coordinates[i][0], coordinates[i][1]] for i in route]
    folium.PolyLine(points, color=color, weight=5, opacity=0.9).add_to(after_map)
    for node in route[1:-1]:
        order_id = stops.iloc[node - 1]["Order_ID"]
        folium.CircleMarker(
            [coordinates[node][0], coordinates[node][1]],
            radius=4,
            color=color,
            fill=True,
            fill_opacity=0.9,
            popup=order_id,
        ).add_to(after_map)

after_map


## Results

Headline miles are the geodesic totals above, rounded to 0.1 mile. The annual figure uses those rounded miles so the arithmetic matches the printed formula: miles saved × $1.20 × 365.

That 365-day factor is the project's stated assumption. It is not a measured year. These 100 orders fall on 40 different dates.


In [9]:
baseline_reported = round(baseline_miles, 1)
optimized_reported = round(optimized_miles, 1)
miles_saved = round(baseline_reported - optimized_reported, 1)
percent_saved = miles_saved / baseline_reported * 100
annual_savings = miles_saved * COST_PER_MILE * DAYS_PER_YEAR

print("FINAL RESULTS")
print(f"Stops in both totals     : {len(stops)}")
print(f"Before optimization      : {baseline_reported:,.1f} miles")
print(f"After optimization       : {optimized_reported:,.1f} miles")
print(f"Miles saved              : {miles_saved:,.1f} miles")
print(f"Savings                  : {percent_saved:.1f}%")
print(f"Vans used                : {vans_used}")
print(f"Stops-per-van cap        : {MAX_STOPS_PER_VAN}")
print(f"Cost assumption          : ${COST_PER_MILE:.2f} per mile")
print(f"Annualization            : x {DAYS_PER_YEAR} days")
print(f"Estimated annual savings : ${annual_savings:,.2f}")
print(
    f"Formula                  : {miles_saved:,.1f} x ${COST_PER_MILE:.2f} x {DAYS_PER_YEAR} "
    f"= ${annual_savings:,.2f}"
)


FINAL RESULTS
Stops in both totals     : 90
Before optimization      : 45,517.9 miles
After optimization       : 7,858.1 miles
Miles saved              : 37,659.8 miles
Savings                  : 82.7%
Vans used                : 6
Stops-per-van cap        : 15
Cost assumption          : $1.20 per mile
Annualization            : x 365 days
Estimated annual savings : $16,494,992.40
Formula                  : 37,659.8 x $1.20 x 365 = $16,494,992.40
